# Imputation Model Training

Trains and serialises an imputation pipeline to fill missing environmental variables in incoming farm profiles.

## Approach: IterativeImputer + KNN

Uses scikit-learn `IterativeImputer` with `KNeighborsRegressor` as the base estimator.
Each feature is modelled as a function of all others and the process iterates to convergence —
more robust than a fixed sequential pipeline.

`StandardScaler` is applied before imputation because KNN is distance-based and
unscaled features (e.g. `elevation_m` range 9–1313) would dominate over others (e.g. `ph` range 5.6–8.5).

## Features

| Column | Type | Notes |
|---|---|---|
| `elevation_m` | Numeric | |
| `slope` | Numeric | |
| `temperature_celsius` | Numeric | |
| `rainfall_mm` | Numeric | |
| `soil_texture_id` | Numeric (int FK) | Rounded to nearest valid ID post-imputation |
| `ph` | Numeric | |

Base features always available from farm geometry: `latitude`, `longitude`, `area_ha`, `coastal`, `riparian`.

## Dataset

`backend/src/scripts/data/farm_master.csv` — 3200 rows, no missing values.
The imputer is trained on the full dataset (learning feature relationships).
Evaluation is done by artificially masking 20% of target values (MCAR) and measuring recovery.

## 1. Imports

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
DATA_PATH = Path("../../backend/src/scripts/data/farm_master.csv")
MODELS_PATH = Path("../src/models/imputation")
MODELS_PATH.mkdir(parents=True, exist_ok=True)

## 2. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Nulls: {df.isnull().sum().sum()}")
print()
print(df.dtypes)
df.head()

## 3. Prepare Features

In [ ]:
# All features used for imputation (base + targets)
BASE_FEATURES = ["latitude", "longitude", "area_ha", "coastal", "riparian"]
TARGET_FEATURES = ["elevation_m", "slope", "temperature_celsius", "rainfall_mm", "soil_texture_id", "ph"]
ALL_FEATURES = BASE_FEATURES + TARGET_FEATURES

# Valid soil_texture_id values (FK to lookup table) — used to snap regression output to valid ID
VALID_SOIL_IDS = sorted(df["soil_texture_id"].unique().tolist())
print("Valid soil_texture_id values:", VALID_SOIL_IDS)

# Prepare feature matrix — convert booleans to int for sklearn
X_full = df[ALL_FEATURES].copy()
X_full["coastal"] = X_full["coastal"].astype(int)
X_full["riparian"] = X_full["riparian"].astype(int)

print(f"Feature matrix shape: {X_full.shape}")
X_full.head()

## 4. Evaluate via Artificial Masking

Since the dataset has no missing values, we simulate missingness (MCAR — missing completely at random)
on a 20% holdout set. We mask 30% of target feature values, run the imputer, and measure recovery.

This validates that the pipeline can recover realistic values before fitting the final model on all data.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

X_train_eval, X_test_eval = train_test_split(X_full, test_size=0.2, random_state=RANDOM_SEED)

# Build and fit evaluation pipeline on training split
eval_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("imputer", IterativeImputer(
        estimator=KNeighborsRegressor(n_neighbors=5),
        max_iter=10,
        random_state=RANDOM_SEED,
    )),
])
eval_pipe.fit(X_train_eval)

# Mask 30% of target values in the test split
MASK_RATE = 0.30
X_masked = X_test_eval.copy()
true_values = {}

for col in TARGET_FEATURES:
    mask = rng.random(len(X_masked)) < MASK_RATE
    true_values[col] = X_masked.loc[mask, col].values.copy()
    X_masked.loc[mask, col] = np.nan

print(f"Masked {MASK_RATE:.0%} of each target column in {len(X_masked)} test rows")
print(f"Total masked cells: {X_masked[TARGET_FEATURES].isnull().sum().sum()}")

In [ ]:
# Impute masked values
X_imputed = eval_pipe.transform(X_masked)
X_imputed_df = pd.DataFrame(X_imputed, columns=X_full.columns, index=X_masked.index)

# Snap soil_texture_id to nearest valid FK value
def snap_to_valid(val, valid_ids):
    return min(valid_ids, key=lambda v: abs(v - val))

X_imputed_df["soil_texture_id"] = X_imputed_df["soil_texture_id"].apply(
    lambda v: snap_to_valid(v, VALID_SOIL_IDS)
)

# Evaluate recovery for each target
header = f"{'Target':25s} | {'RMSE':>8} | {'MAE':>8} | {'R2':>6}"
print(header)
print("-" * 58)
eval_results = {}
for col in TARGET_FEATURES:
    mask = X_masked[col].isnull()
    if mask.sum() == 0:
        continue
    y_true = true_values[col]
    y_pred = X_imputed_df.loc[mask, col].values

    if col == "soil_texture_id":
        acc = accuracy_score(y_true.astype(int), y_pred.astype(int))
        eval_results[col] = {"Accuracy": round(acc, 3)}
        print(f"{col:25s} | Accuracy: {acc:.3f}  (n={mask.sum()})")
    else:
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        eval_results[col] = {"RMSE": round(rmse, 3), "MAE": round(mae, 3), "R2": round(r2, 3)}
        print(f"{col:25s} | {rmse:8.3f} | {mae:8.3f} | {r2:6.3f}  (n={mask.sum()})")

## 5. Fit Final Pipeline on Full Dataset

In [ ]:
# Fit on all 3200 rows for production use
final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("imputer", IterativeImputer(
        estimator=KNeighborsRegressor(n_neighbors=5),
        max_iter=10,
        random_state=RANDOM_SEED,
    )),
])
final_pipe.fit(X_full)

# Save the full pipeline (scaler + imputer together)
joblib.dump(final_pipe, MODELS_PATH / "imputation_pipeline.joblib")
joblib.dump(VALID_SOIL_IDS, MODELS_PATH / "valid_soil_texture_ids.joblib")
joblib.dump(list(X_full.columns), MODELS_PATH / "feature_columns.joblib")

print("Saved:")
for f in sorted(MODELS_PATH.glob("*.joblib")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size_kb:6.1f} KB")

## 6. Usage Example

How the imputation service should use the saved pipeline:

```python
import joblib
import numpy as np
import pandas as pd

pipe = joblib.load("src/models/imputation/imputation_pipeline.joblib")
valid_soil_ids = joblib.load("src/models/imputation/valid_soil_texture_ids.joblib")
feature_columns = joblib.load("src/models/imputation/feature_columns.joblib")

# Incoming farm profile with some missing values
farm = {
    "latitude": -8.57, "longitude": 126.68, "area_ha": 1.2,
    "coastal": 0, "riparian": 0,
    "elevation_m": np.nan, "slope": np.nan,
    "temperature_celsius": 24.0, "rainfall_mm": np.nan,
    "soil_texture_id": np.nan, "ph": 6.5,
}

X = pd.DataFrame([farm])[feature_columns]
X_imputed = pipe.transform(X)
result = dict(zip(feature_columns, X_imputed[0]))

# Snap soil_texture_id to nearest valid FK
result["soil_texture_id"] = min(valid_soil_ids, key=lambda v: abs(v - result["soil_texture_id"]))
print(result)
```